# Clase 4 · 01 · El log: topics, offsets y consumidores independientes

En la teoría vimos que Kafka organiza los eventos en un **log particionado**: una secuencia *append-only* donde cada mensaje tiene una posición (**offset**). Los consumidores no borran los mensajes; cada uno recuerda **hasta qué offset leyó**.

Databricks Free Edition no trae Kafka, pero una **tabla Delta** se comporta de forma muy parecida:

| Kafka | Tabla Delta leída como stream |
|---|---|
| Topic `pagos` | Tabla `stream_topic_payments` |
| Productor publica mensajes | `append` de filas nuevas (cada append crea una **versión**) |
| Offset del consumidor | Versión de la tabla guardada en el **checkpoint** |
| Consumer group distinto | Consulta con **otro checkpoint** |
| Retención: el mensaje no se borra al leerlo | Las filas siguen en la tabla |
| Resetear el offset y reprocesar (*replay*) | Nueva consulta que lee desde la versión 0 |

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_rules

In [ ]:
%run ../common/streaming_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
for table in [customers, products]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2."
T = lambda name: qualified_table(config, name)

In [ ]:
LAB = "log"
reset_lab(spark, dbutils, config, LAB, ["stream_topic_payments", "stream_consumer_fraud", "stream_consumer_reports", "stream_consumer_replay"])
topic = T("stream_topic_payments")
spark.sql(f"""CREATE TABLE {topic} (event_id STRING, event_ts TIMESTAMP, customer_id BIGINT,
              amount DOUBLE, payment_channel STRING, source_batch_id STRING) USING DELTA""")
print("Topic creado (versión 0, vacío):", topic)

## 1. Cola tradicional vs. log

Primero, en Python puro: en una **cola** (RabbitMQ, JMS) leer un mensaje lo **borra**. En un **log** (Kafka) leer no modifica nada; el consumidor sólo avanza su offset.

In [ ]:
from collections import deque

messages = [f"pago_{i}" for i in range(1, 6)]
queue = deque(messages)
consumed_from_queue = [queue.popleft() for _ in range(3)]      # el consumidor lee 3 mensajes

log = list(messages)
offset = 0
consumed_from_log = log[offset:offset + 3]
offset += 3                                                      # sólo avanza su posición

print(f"Cola → leídos {consumed_from_queue}; quedan en la cola: {list(queue)}")
print(f"Log  → leídos {consumed_from_log}; el log sigue teniendo {len(log)} mensajes; offset del consumidor = {offset}")

## 2. Un productor y dos consumidores

- El **productor** publica las cinco llegadas, una por vez.
- El consumidor **`antifraude`** es rápido: procesa después de *cada* llegada.
- El consumidor **`reportes`** es lento: sólo procesa después de las llegadas 3 y 5.

Cada consumidor tiene **su propio checkpoint** (como consumer groups distintos en Kafka), así que los dos reciben todos los mensajes (*fan-out*) sin molestarse entre sí.

In [ ]:
def publish_to_topic(batch_id):
    rows = build_stream_rows(config, batch_id)          # todos los mensajes que envía el productor
    frame = (spark.createDataFrame(pd.DataFrame(rows))
             .select("event_id", F.col("event_ts").cast("timestamp").alias("event_ts"),
                     F.expr("try_cast(customer_id AS BIGINT)").alias("customer_id"),
                     F.expr("try_cast(amount AS DOUBLE)").alias("amount"), "payment_channel", "source_batch_id"))
    frame.write.format("delta").mode("append").saveAsTable(topic)
    return len(rows)

def consume(name, sink):
    stream = (spark.readStream.table(topic)
              .withColumn("consumer", F.lit(name))
              .withColumn("consumed_at", F.current_timestamp()))
    cp = lab_path(config, LAB, "checkpoints", name)
    new_rows = rows_added(spark, sink, lambda: run_stream(stream, cp, table=sink))
    return new_rows, checkpoint_offset(dbutils, cp)

def topic_version():
    return spark.sql(f"DESCRIBE HISTORY {topic}").agg(F.max("version")).first()[0]

timeline = []
for step, batch_id in enumerate(["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], start=1):
    published = publish_to_topic(batch_id)
    read_fraud, offset_fraud = consume("antifraude", T("stream_consumer_fraud"))
    if step in (3, 5):
        read_reports, offset_reports = consume("reportes", T("stream_consumer_reports"))
    else:
        read_reports, offset_reports = 0, None
    timeline.append({
        "llegada": step, "publicados": published, "versión del topic": topic_version(),
        "mensajes en el topic": spark.table(topic).count(),
        "leídos por antifraude (total)": spark.table(T("stream_consumer_fraud")).count(),
        "leídos por reportes (total)": spark.table(T("stream_consumer_reports")).count() if spark.catalog.tableExists(T("stream_consumer_reports")) else 0,
    })
    print(f"llegada {step}: antifraude leyó {read_fraud} mensajes nuevos; reportes leyó {read_reports}")

timeline = pd.DataFrame(timeline)
display(timeline)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(timeline["llegada"], timeline["mensajes en el topic"], lw=9, alpha=0.35, color="#64748b", label="mensajes en el topic (log)")
ax.plot(timeline["llegada"], timeline["leídos por antifraude (total)"], marker="o", color="#16a34a", label="consumidor antifraude")
ax.step(timeline["llegada"], timeline["leídos por reportes (total)"], where="post", marker="s", color="#f59e0b", label="consumidor reportes")
ax.set_xlabel("Llegada")
ax.set_ylabel("Mensajes")
ax.set_xticks(timeline["llegada"])
ax.set_title("Cada consumidor avanza a su ritmo; el log no se vacía")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Para observar:** después de la llegada 2, `reportes` está **atrasado** (no leyó nada), pero eso no afecta a `antifraude` ni al topic. Cuando `reportes` vuelve a ejecutarse, retoma desde su checkpoint y se pone al día. Los mensajes del topic nunca disminuyen: leer no borra.

## 3. ¿Dónde está el offset?

El **checkpoint** de cada consulta guarda hasta qué **versión** del topic leyó (su offset). Ese dato es lo que permite retomar donde quedó.

In [ ]:
for name in ["antifraude", "reportes"]:
    sink = T("stream_consumer_fraud") if name == "antifraude" else T("stream_consumer_reports")
    rows_read, offset = consume(name, sink)       # reejecutar sin mensajes nuevos
    print(f"{name:10s} → mensajes nuevos al reejecutar: {rows_read} · offset en el checkpoint (próxima versión a leer): {offset}")
print(f"Última versión del topic: {topic_version()}")

Al reejecutar sin mensajes nuevos, ningún consumidor vuelve a leer nada: el offset guardado en el checkpoint ya apunta al final del log: indica la **próxima** versión a leer (una más que la última versión del topic).

## 4. Replay: reprocesar todo con otra lógica

Supongamos que cambiamos la lógica de reportes: ahora queremos el **monto total por canal**. En una arquitectura **Lambda** habría dos sistemas: un batch preciso y una capa de streaming rápida, con la misma lógica escrita dos veces. En **Kappa** hay un solo código de streaming: para reprocesar se lanza una consulta nueva (otro checkpoint) que lee el log **desde el principio**.

In [ ]:
replay = (spark.readStream.option("startingVersion", 0).table(topic)
          .select("payment_channel", "amount", "source_batch_id"))
progress = run_stream(replay, lab_path(config, LAB, "checkpoints", "replay_v2"), table=T("stream_consumer_replay"))
print(f"El replay leyó {spark.table(T('stream_consumer_replay')).count()} mensajes (el topic tiene {spark.table(topic).count()})")
display(spark.table(T("stream_consumer_replay")).groupBy("payment_channel")
        .agg(F.count("*").alias("pagos"), F.round(F.sum("amount"), 2).alias("monto_total"))
        .orderBy("payment_channel"))

## Resumen

- El log **conserva** los mensajes; cada consumidor guarda su **offset**.
- Un consumidor lento sólo se atrasa a sí mismo (**aislamiento**).
- Varios consumidores con checkpoints distintos reciben todos los mensajes (**fan-out**).
- Con el log completo se puede **reprocesar** desde el inicio con lógica nueva (**replay**, arquitectura Kappa).